In [ ]:
import pandas as pd
from pathlib import Path
import openmeteo_requests
import requests_cache
from retry_requests import retry


def openmeteo_hourly_to_parquet(
    responses,
    hourly_vars,
    save_path="../data/processed/openmeteo_hourly.parquet",
):
    """
    Universal Open-Meteo hourly parser
    """

    cities = [
        "Zurich", "Geneva", "Bern", "Basel", "Lausanne",
        "Lucerne", "St_Gallen", "Lugano", "Interlaken", "Central_CH"
    ]

    dfs = []

    for i, response in enumerate(responses):

        hourly = response.Hourly()

        data = {
            "timestamp_utc": pd.date_range(
                start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=hourly.Interval()),
                inclusive="left",
            ),
            "city": cities[i],
            "location_id": i,
            "latitude": response.Latitude(),
            "longitude": response.Longitude(),
            "elevation": response.Elevation(),
            "retrieved_at_utc": pd.Timestamp.utcnow(),
        }

        for j, var in enumerate(hourly_vars):
            data[var] = hourly.Variables(j).ValuesAsNumpy()

        dfs.append(pd.DataFrame(data))

    full_df = pd.concat(dfs, ignore_index=True)

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    full_df.to_parquet(save_path, index=False)

    print(f"Saved: {save_path}")
    print(full_df.shape)

    return full_df


# ==========================================
# API CLIENT
# ==========================================

cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

LAT = [47.3769, 46.2044, 46.9480, 47.5596, 46.5197,
       47.0502, 47.4245, 46.0101, 46.6863, 46.8182]

LON = [8.5417, 6.1432, 7.4474, 7.5886, 6.6323,
       8.3093, 9.3767, 8.9600, 7.8632, 8.2275]

HOURLY_VARS = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation_probability",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "cloud_cover",
    "wind_speed_10m",
    "surface_pressure",
    "is_day",
    "sunshine_duration",
]


# ==========================================
# 1. HISTORICAL ACTUAL WEATHER
# ==========================================




# ==========================================
# 2. HISTORICAL FORECAST WEATHER
# ==========================================



# ==========================================
# 3. LIVE FORECAST WEATHER (next 24h)
# ==========================================



KeyboardInterrupt: 

In [16]:
archive_url = "https://archive-api.open-meteo.com/v1/archive"

archive_params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": "2020-01-01",
    "end_date": "2026-03-30",
    "hourly": HOURLY_VARS,
    "timezone": "Europe/Zurich",
}

responses = openmeteo.weather_api(archive_url, params=archive_params)

historical_df = openmeteo_hourly_to_parquet(
    responses,
    HOURLY_VARS,
    "../data/processed/swiss_weather_historical.parquet",
)

Saved: ../data/processed/swiss_weather_historical.parquet
(547440, 20)


In [24]:
hist_fcst_url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

hist_fcst_params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": "2020-01-01",
    "end_date": "2026-03-30",
    "hourly": HOURLY_VARS,
    "timezone": "Europe/Zurich",
    "models": "best_match",
}

responses = openmeteo.weather_api(hist_fcst_url, params=hist_fcst_params)

historical_forecast_df = openmeteo_hourly_to_parquet(
    responses,
    HOURLY_VARS,
    "../data/processed/swiss_weather_historical_forecast.parquet",
)


OpenMeteoRequestsError: failed to request 'https://historical-forecast-api.open-meteo.com/v1/forecast': {'reason': 'Hourly API request limit exceeded. Please try again in the next hour.', 'error': True}

In [ ]:
forecast_url = "https://api.open-meteo.com/v1/forecast"

forecast_params = {
    "latitude": LAT,
    "longitude": LON,
    "hourly": HOURLY_VARS,
    "forecast_hours": 24,
    "timezone": "Europe/Zurich",
}

responses = openmeteo.weather_api(forecast_url, params=forecast_params)

forecast_df = openmeteo_hourly_to_parquet(
    responses,
    HOURLY_VARS,
    "../data/processed/swiss_weather_live_forecast.parquet",
)